<a href="https://colab.research.google.com/github/DariaLaska/ml/blob/main/%D0%A1%D0%B5%D0%B3%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D1%86%D0%B8%D1%8F_%D1%82%D0%B5%D0%BA%D1%81%D1%82%D0%B0_%D0%B4%D0%BE%D0%BF_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymorphy3
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 81.9 MB/s eta 0:00:00


In [ ]:
!wget https://dataset.role.ru/apartment-descriptions.zip
!unzip -q apartment-descriptions.zip

--2025-12-14 23:11:45--  https://dataset.role.ru/apartment-descriptions.zip
Resolving dataset.role.ru (dataset.role.ru)... 176.115.205.154
Connecting to dataset.role.ru (dataset.role.ru)|176.115.205.154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66144 (65K) [application/zip]
Saving to: ‘apartment-descriptions.zip’

apartment-descripti 100%[===================>]  64.59K   372KB/s    in 0.2s    

2025-12-14 23:11:46 (372 KB/s) - ‘apartment-descriptions.zip’ saved [66144/66144]



In [ ]:
import re
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

def normalize_word(word):
    # Удаление знаков препинания
    cleaned = re.sub(r'[^\w\s]', '', word).lower()
    # Лемматизация
    if cleaned:
        lemma = morph.parse(cleaned)[0].normal_form
        return lemma
    return ""

# Пример использования
word = "Бывает:;;"
normalized = normalize_word(word)
print(normalized)

бывать


In [ ]:
pattern = re.compile(r'<([^>]+)>(.*?)</\1>', flags=re.S)
tag_rename = {'s1': 'квартира', 's2': 'жк', 's3': 'расположение', 's4': 'цена'}

def normalize_text_for_training(text):
    X_train = []
    y_train = []
    last_pos = 0

    for match in pattern.finditer(text):
        # Текст до тега
        before_tag = text[last_pos:match.start()]
        if before_tag.strip():
            for word in before_tag.split():
                normalized_word = normalize_word(word)
                if normalized_word:
                    X_train.append(normalized_word)
                    y_train.append('O')

        # Текст внутри тега
        tag = match.group(1)
        tag = tag_rename.get(tag, tag)
        inside_text = match.group(2)
        words = inside_text.split()
        if words:
            for i, word in enumerate(words):
                normalized_word = normalize_word(word)
                if normalized_word:
                    X_train.append(normalized_word)
                    y_train.append(f'{tag}' if i == 0 else f'{tag}')

        last_pos = match.end()

    # Текст после последнего тега
    after_last_tag = text[last_pos:]
    if after_last_tag.strip():
        for word in after_last_tag.split():
            normalized_word = normalize_word(word)
            if normalized_word:
                X_train.append(normalized_word)
                y_train.append('O')

    return X_train, y_train

# Пример использования
text = "Текст, в <class1>нём первый класс</class1> <class2> в нём второй класс</class2> и так далее"
X_train, y_train = normalize_text_for_training(text)

print("X_train:", X_train)
print("y_train:", y_train)

X_train: ['текст', 'в', 'он', 'первый', 'класс', 'в', 'он', 'второй', 'класс', 'и', 'так', 'далее']
y_train: ['O', 'O', 'class1', 'class1', 'class1', 'class2', 'class2', 'class2', 'class2', 'O', 'O', 'O']


In [ ]:
import os

dataset_dir = "/content/apartment-descriptions"

texts, markup = [], []

for description_file in os.listdir(dataset_dir):
    file_path = os.path.join(dataset_dir, description_file)
    with open(file_path, mode='r', encoding='utf-8') as f:
        content = f.read()
        norm_text, classes = normalize_text_for_training(content)

        texts.append(norm_text)
        markup.append(classes)

In [ ]:
from gensim.models import Word2Vec
import numpy as np

word2vec_model = Word2Vec(texts, vector_size=100, window=5, min_count=1, workers=4)

def get_word_embedding(word):
    return word2vec_model.wv[word]

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

unique_classes = sorted(set([cls for classes in markup for cls in classes]))
class_to_idx = {cls: idx for idx, cls in enumerate(unique_classes)}
print(class_to_idx)
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(np.array(list(class_to_idx.keys())).reshape(-1, 1))

def get_onehot_encoding(cls):
    return encoder.transform([[cls]])[0]

{'O': 0, 'жк': 1, 'квартира': 2, 'расположение': 3, 'цена': 4}


In [ ]:
import pandas as pd

# Подготовка X_train и y_train
X_train = []
y_train = []

# classes_table = pd.DataFrame()

classes_count = {key: 0 for key in unique_classes}
classes_count['a'] = 0

for index, (sentence, classes) in enumerate(zip(texts, markup)):
    # sentence_table = pd.DataFrame({f'word{index}': sentence, f'class{index}': classes})
    # classes_table = pd.concat([classes_table, sentence_table], axis=1)

    for word, cls in zip(sentence, classes):
        word_embedding = get_word_embedding(word)
        X_train.append(word_embedding)

        onehot_vector = get_onehot_encoding(cls)
        y_train.append(onehot_vector)

        classes_count[cls] += 1
        classes_count['a'] += 1


# Преобразуем в numpy массивы
X_train = np.array(X_train)
y_train = np.array(y_train)

# display(classes_table.head(200))

print(X_train.shape)
print(y_train.shape)
print(classes_count)

(7924, 100)
(7924, 5)
{'O': 4076, 'жк': 1220, 'квартира': 1365, 'расположение': 980, 'цена': 283, 'a': 7924}


In [ ]:
import numpy as np

# Найдем максимальную длину последовательности в X_train
max_seq_length = 96
print(f"Максимальная длина последовательности: {max_seq_length}")

# Паддинг для X_train и y_train
def pad_sequences_to_length(sequences, max_length, padding_value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) < max_length:
            # Дополняем последовательность до max_length
            padded_seq = np.vstack([seq, np.tile(padding_value, (max_length - len(seq), seq.shape[1]))])
        else:
            # Обрезаем до max_length
            padded_seq = seq[:max_length]
        padded_sequences.append(padded_seq)
    return np.array(padded_sequences)

# Преобразуем X_train и y_train в последовательности
X_train_sequences = []
y_train_sequences = []

# Группируем данные в последовательности длиной max_seq_length
for i in range(0, len(X_train) - max_seq_length + 1, max_seq_length):
    X_seq = X_train[i:i + max_seq_length]
    y_seq = y_train[i:i + max_seq_length]
    X_train_sequences.append(X_seq)
    y_train_sequences.append(y_seq)

# Паддинг для последней последовательности, если она короче
if len(X_train) % max_seq_length != 0:
    last_X_seq = X_train[-max_seq_length:]
    last_y_seq = y_train[-max_seq_length:]
    X_train_sequences.append(last_X_seq)
    y_train_sequences.append(last_y_seq)

# Преобразуем в numpy массивы
X_train_padded = np.array(X_train_sequences)
y_train_padded = np.array(y_train_sequences)

print("X_train_padded shape:", X_train_padded.shape)
print("y_train_padded shape:", y_train_padded.shape)

Максимальная длина последовательности: 96
X_train_padded shape: (83, 96, 100)
y_train_padded shape: (83, 96, 5)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, Concatenate, TimeDistributed, BatchNormalization, UpSampling1D, Activation, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, Concatenate, TimeDistributed
from tensorflow.keras.optimizers import Adam

# Параметры
embedding_dim = X_train.shape[1]  # Размерность векторов эмбеддингов
num_classes = y_train.shape[1]    # Количество классов

input_layer = Input(shape=(max_seq_length, embedding_dim))

# CNN ветка
cnn_layer = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(input_layer)
cnn_layer = MaxPooling1D(pool_size=1)(cnn_layer)
cnn_layer = Conv1D(filters=256, kernel_size=3, activation='relu', padding='same')(cnn_layer)
cnn_layer = MaxPooling1D(pool_size=1)(cnn_layer)
cnn_layer = Dropout(0.3)(cnn_layer)  # Уменьшил dropout для переобучения

# BiLSTM ветка
bilstm_layer = Bidirectional(LSTM(units=128, return_sequences=True))(cnn_layer)
bilstm_layer = Bidirectional(LSTM(units=64, return_sequences=True))(bilstm_layer)
bilstm_layer = Dropout(0.3)(bilstm_layer)  # Уменьшил dropout

# Полносвязный слой
output_layer = TimeDistributed(Dense(num_classes, activation='softmax'))(bilstm_layer)

model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Callbacks
# 1. Early Stopping: stops training when a monitored quantity has stopped improving.
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=15,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity.
)

# 2. Model Checkpoint: saves the model after every epoch if the monitored quantity has improved.
# To save to Google Drive, make sure to mount it first: from google.colab import drive; drive.mount('/content/drive')
model_checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/МоЛоко/weights/text_sega.keras', # Path to save the model. e.g., '/content/drive/MyDrive/best_model.h5'
    monitor='val_loss',       # Monitor validation loss
    save_best_only=True,      # Save only the best model
    verbose=0                 # Log when a model is saved
)

# 3. Reduce Learning Rate on Plateau: reduces learning rate when a metric has stopped improving.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor validation loss
    factor=0.2,          # Factor by which the learning rate will be reduced. new_lr = lr * factor
    patience=3,          # Number of epochs with no improvement after which learning rate will be reduced
    min_lr=1e-6,      # Lower bound on the learning rate.
    verbose=0            # Log when learning rate is reduced
)

callbacks = [model_checkpoint, reduce_lr]

# Обучаем модель
history = model.fit(
    X_train_padded,
    y_train_padded,
    batch_size=4,
    epochs=100,
    validation_split=0.2,
    callbacks=callbacks # Pass the list of callbacks here
)

Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 14s 154ms/step - accuracy: 0.4379 - loss: 1.4901 - val_accuracy: 0.6238 - val_loss: 1.2056 - learning_rate: 0.0010
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.4702 - loss: 1.3805 - val_accuracy: 0.6238 - val_loss: 1.1627 - learning_rate: 0.0010
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.4783 - loss: 1.3630 - val_accuracy: 0.6238 - val_loss: 1.1645 - learning_rate: 0.0010
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5017 - loss: 1.2716 - val_accuracy: 0.6293 - val_loss: 1.1396 - learning_rate: 0.0010
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5135 - loss: 1.2151 - val_accuracy: 0.6330 - val_loss: 1.0520 - learning_rate: 0.0010
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5510 - loss: 1.1529 - val_accuracy: 0.6127 - val_loss: 1.0771 - learning_rate: 0.0010
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.5353 - loss: 1.1920 

In [ ]:
def prepare_text_for_inference(text, word2vec_model, encoder):
    # Нормализация текста и получение X (слов) и y (классов)
    X, _ = normalize_text_for_training(text)

    # Получение эмбеддингов для каждого слова
    X_embeddings = []
    for word in X:
        try:
            embedding = get_word_embedding(word)
        except KeyError:
            # Если слово отсутствует в модели, используем нулевой вектор
            embedding = np.zeros(word2vec_model.vector_size)
        X_embeddings.append(embedding)

    # Паддинг последовательностей
    X_padded = pad_sequences_to_length([np.array(X_embeddings)], max_seq_length)

    return X_padded, X  # Возвращаем и X_padded, и X (нормализованные слова)

def predict_classes(model, X_padded):
    # Предсказание классов
    y_pred = model.predict(X_padded)
    # Получение индексов классов с максимальной вероятностью
    y_pred_classes = np.argmax(y_pred, axis=-1)
    return y_pred_classes

def format_predicted_text(text, X, y_pred_classes, idx_to_class):
    predicted_classes = [idx_to_class[idx] for idx in y_pred_classes[0]]

    result = []
    i = 0
    n = min(len(X), len(predicted_classes))

    while i < n:
        word = X[i]
        cls = predicted_classes[i]
        if cls != 'O':
            result.append(f'<{cls}>')
            result.append(word)
            i += 1
            # Продолжаем добавлять слова с тем же классом
            while i < n and predicted_classes[i] == cls:
                result.append(X[i])
                i += 1
            result.append(f'</{cls}>')
        else:
            result.append(word)
            i += 1

    return ' '.join(result)

In [ ]:
# Пример использования
new_text = """
Продается однокомнатный апартамент 181 в новостройке Квартал "Зорге 9" по адресу Москва, САО, Хорошевский, ул. Зорге, д. 9, корп. 2 Общая площадь апартамента - 36.35 кв. м., этаж 17 из 21, секция 1. Тип проекта, по которому построен дом - монолит, компания-застройщик - St. Michael. Стоимость квартиры - 8828922 Р. Срок сдачи - 2 квартал 2022 года. Более подробная информация по телефону. "Зорге 9" - это комплекс бизнес-класса, расположенный на Ходынке - одном из самых престижных и востребованных районов Москвы. Жилой квартал окружен современными спортивными аренами, школами, ВУЗами, магазинами, салонами красоты и многими другими объектами инфраструктуры. Неподалеку находится Березовая роща и парк "Ходынское поле", где можно прогуляться и отдохнуть от городской суеты, а также один из крупнейших торгово-развлекательных центров Москвы - "Авиапарк". Добраться до комплекса можно от станции метро "Полежаевская" и станции МЦК "Зорге" пешком за 10 минут. Действуют выгодные предложения. Звоните!
"""

X_padded, X = prepare_text_for_inference(new_text, word2vec_model, encoder)
y_pred_classes = predict_classes(model, X_padded)

# Создаем словарь для обратного преобразования индексов в классы
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

formatted_text = format_predicted_text(new_text, X, y_pred_classes, idx_to_class)
print(formatted_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
продаваться однокомнатный апартамент 181 в новостройка <квартира> квартал зорг 9 по адрес москва </квартира> <жк> сао хорошевский ул </жк> <квартира> зорг далее 9 корп 2 общий площадь апартамент 3635 кв м этаж 17 из 21 секция 1 тип проект по </квартира> <жк> который построить дом монолит компаниязастройщик st michael стоимость </жк> <квартира> квартира 8828922 р срок сдача 2 квартал 2022 год более подробный информация по телефон зорг 9 это комплекс бизнескласс расположить на ходынка один из </квартира> самый престижный и востребовать район москва жилой квартал окружный современный спортивный арен школа вуз магазин салон красота и многий другой объект инфраструктура неподалёку находиться берёзовый роща и парк ходынский
